# Data Aggregation and Group Operations

References:

[1] McKinney, Wes. *Python for data analysis.* " O'Reilly Media, Inc.", 2022.

[2] VanderPlas, Jake. *Python data science handbook: Essential tools for working with data. Second Edition* " O'Reilly Media, Inc.", 2023.

[3] Johansson, Robert, Robert Johansson, and Suresh John. *Numerical python.* Vol. 1. New York: Apress, 2019.

[4] pandas - Groupby: split-apply-combine https://pandas.pydata.org/docs/user_guide/groupby.html

Categorizing a dataset, creating groups, and applying a funciton to each group, whether an aggregation or transformation, is a critical component of data analysis workflow. In this notebook, we will explore how we can use group operations using pandas to perform such procedures.


In [ ]:
import numpy as np
import pandas as pd

## 1 Group Operations

To illustrate how group operations work, we consider a small tabular dataset which we defined as a `DataFrame`:

In [ ]:
rng = np.random.default_rng(seed=143)

df = pd.DataFrame({'g1': ['a', 'a', None, 'b', 'b', 'a', None],
                   'g2': pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
                   'data1': rng.integers(10, size=7),
                   'data2': rng.integers(10, size=7)})
df

Suppose that we wanted to compute the mean of the `data1` column with respect to the labels of `g1` column. One way is to access the `data1`, then call `groupby` with the column `g1`.

In [ ]:
grouped = df['data1'].groupby(df['g1'])
grouped

The grouped variable is a `GroupBy` objet which has not done any computation yet, but the obejct has all the information needed to apply any group operations to the desiged grouop as specified during the `groupby` call. For example, if we want to get the mean of each group, we can call the `mean` method of `GroupBy`.

In [ ]:
grouped.mean()

If instead, we passed multiple arrays as a list, we'd get:

In [ ]:
grouped = df['data1'].groupby([df['g1'], df['g2']])
grouped

In [ ]:
means = grouped.mean()
means

We grouped the data using two keys and the resulting `Series` has hierarchical index. We can work with the Series having a hierarchical index but usually it is convenient to transform it back into a `DataFrame` which we can do in two ways:

In [ ]:
means.reset_index()

In [ ]:
means.unstack()

When grouping is performed on a dataframe, aggregation can be done across all possible columns. For example,

In [ ]:
df

In [ ]:
df.groupby('g2').mean(numeric_only=True)

Notice that `df['g1']` was not included in the column since it is not numeric data. In this case, it is said to be a *nuisance* column and pandas automatically excludes it from the result.

A generally useful `GroupBy` method is the `size`, which returns a `Series` containing the group's size (not counting the nulls).

In [ ]:
df.groupby(['g1', 'g2']).size()

You can also select a sequence of columns to subset the group operation along those columns,

In [ ]:
df

In [ ]:
df.groupby('g1')['data1'].mean()

You can also iterate over a group. A `GroupBy` object generates a sequence of 2-tuples containing the group name along with the associated chunk of data.

In [ ]:
for name, group in df.groupby('g1'):
    print(name)
    print(group)

You can also specify the grouping via dictionaries or series, functions, and index levels. We show them briefly on the next few cells.

In [ ]:
rng = np.random.default_rng(143)

people = pd.DataFrame(rng.integers(5, size=(5, 5)),
                      columns=["a", "b", "c", "d", "e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])
people

Grouping by dictionary

In [ ]:
col_map = {'a': 'red', 'b': 'red', 'c': 'blue', 'd': 'blue', 'e': 'red',
           'f': 'orange'}

In [ ]:
grouped = people.groupby(col_map, axis='columns')
grouped.mean()

In [ ]:
pd.Series(col_map)

In [ ]:
grouped = people.groupby(pd.Series(col_map), axis='columns')
grouped.mean()

Grouping with functions

In [ ]:
people.groupby(len).sum()

Grouping with hierarchical index

In [ ]:
rng = np.random.default_rng(143)
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"],
                                     [1, 3, 5, 1, 3]],
                                    names=["city", "tenor"])
hier_df = pd.DataFrame(rng.integers(5, size=(4, 5)), columns=columns)
hier_df

In [ ]:
hier_df.groupby(level='city', axis='columns').mean()

## 2 Aggregation

*Aggregations* refer to any data transformation that produces scalar values from arrays. Some of the common GroupBy operations are shown below.

<img src="images/aggregations.png" style="width: 55%;">

We've demonstrated some of these methods, let's show how we can also define our own aggregation functions.

To define our own function, we pass any functoin that aggregates an array to the `aggregate` or `agg` method.

In [ ]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

In [ ]:
df

In [ ]:
grouped = df.groupby('g1')
grouped.agg(peak_to_peak)

Let's look at a real world data and how we can perform an aggregation operation by group on it. Consider the NBA 2022-2023 regular season statistics.

In [ ]:
data = pd.read_csv(
    'data/nba-2023-regular.csv', sep=';', encoding = 'ISO-8859-1',
).drop_duplicates(subset='Rk').set_index('Rk')
data

We can group our data according to position and team then aggregate the corresponding group statistics.

In [ ]:
data.columns

In [ ]:
selected_stats = ['3P', '3PA', 'FG%', 'AST', 'BLK', 'TOV']

tm_pos_stats = data.groupby(['Tm', 'Pos'])[selected_stats].agg(
    ['mean', 'std', 'max', 'min', 'count', peak_to_peak]
)
tm_pos_stats

In [ ]:
tm_pos_stats.loc['GSW', '3P']

Upon grouping, you can instead specify a list of (`name`, `function`) tuples in which groupby will use to determine the resulting `DataFrame` columns.

In [ ]:
selected_stats = ['3P', '3PA', 'FG%', 'AST', 'BLK', 'TOV']

tm_pos_stats = data.groupby(['Tm', 'Pos'])[selected_stats].agg(
    [('Average', 'mean'), ('Maximum', 'max'), ('Minimum', 'min'),
     ('Count', 'count'), ('P2P', peak_to_peak)]
)
tm_pos_stats

If you want to apply different aggregation function to one or more columns, you can specify a dictionary that contains a mapping of column names to any of the function specifications listed:

In [ ]:
data.columns

In [ ]:
data.groupby('Pos').agg({'3P': [('Maximum', np.max)], 'STL': [('Average', np.mean)]})

Notice that most of the examples, up until now, the aggregated comes back with an index, which can be hierarchical if we select more than one columns of grouping. We can disable this behavior by setting the `as_index` parameter to `False` during the `groupby` operation.

In [ ]:
tm_pos_stats = data.groupby(['Tm', 'Pos'])[selected_stats].mean()
tm_pos_stats

In [ ]:
tm_pos_stats = data.groupby(['Tm', 'Pos'], as_index=False)[selected_stats].mean()
tm_pos_stats

## 3 Data Transformation

A transformation is a `GroupBy` operation whose result is indexed the same as the one being grouped. The restrictions for a function to be be used in transformation is as follows:

1. It can produce a scalar value to be broadcast to the shape of the group.
2. It can produce an object of the same shape as the input group.
3. It must not mutate its input.

Common examples of transformation operations include `cumsum()` and `diff()`. Consider a data containing beer consumption of users.

In [ ]:
rng = np.random.default_rng(1337)

data = pd.DataFrame({'person': ['Leo', 'K-Ann', 'Basti']*4,
                     'beer_count': rng.integers(4, size=(12))})
data

We can do a cumulative sum across users which counts the amount of beer that they've consumed cumulatively.

In [ ]:
data.groupby('person').cumsum()

Similar to the aggregation method, `transform()` can accept string aliases to the built-in transformation methods or user-defined functions.

In [ ]:
data

In [ ]:
data.groupby('person').beer_count.transform('diff')

We can also define a customize function for transformation:

In [ ]:
def normalize(x):
    return (x - x.mean()) / x.std()

In [ ]:
data.groupby('person').beer_count.transform(normalize)

Notice the difference of aggregation and transformation. Aggregate must produce an aggregated value for each group, meanwhile, transform produces an object of the same length (or broadcastable to the shape of the groups).

In [ ]:
data.groupby('person').beer_count.agg(normalize)

Another common data transformation is to replace missing data with the group mean.

In [ ]:
rng = np.random.default_rng(143)

# Initialize columns and values
cols = ['A', 'B', 'C']
values = rng.random((1_000, 3))

# Add null values
values[rng.integers(1_000, size=100), 0] = np.nan
values[rng.integers(1_000, size=50), 1] = np.nan
values[rng.integers(1_000, size=200), 2] = np.nan

# Initialize dataframe with null values
data = pd.DataFrame(values, columns=cols)
data

In [ ]:
data.isna().sum()

In [ ]:
rng = np.random.default_rng(143)

countries = np.array(['US', 'UK', 'GR', 'JP'])
data['Country'] = countries[rng.integers(4, size=1_000)]
data

In [ ]:
data.groupby('Country').mean()

In [ ]:
data.groupby('Country').transform(lambda x: x.fillna(x.mean()))

## 4 Filtration

Filtration is an operation that subsets the original grouping object. It may filter out entire groups, part of groups, or both. Common groupby operations that act as filtrations are `head()`, `nth()`, and `tail()`.

| Method | Description |
| ------ | ----------- |
| `head()` | Select the top row(s) of each group |
| `nth()` | Select the nth row(s) of each group |
| `tail()` | Select the bottom row(s) of each group |


In [ ]:
data

In [ ]:
data.groupby('Country').head()

In [ ]:
data.groupby('Country').nth([2, 3])

### The `filter` method

If we want to use a User Defined Function (UDF) to perform filtration, we can do so using the `filter` method of `GroupBy`. A requirement for this UDF is that it should return a boolean array which `filter` will use to subset the group for which the UDF is `True`.

As an example, let's say we have a series defined below:

In [ ]:
sf = pd.Series([1, 1, 2, 3, 3, 3])

And we want to take only elements that belong to groups with sum greater than `2`.

In [ ]:
sf

In [ ]:
sf.groupby(sf).filter(lambda x: x.sum() > 2)

Another useful operation is filtering out elements that belong to groups with only a couple of members.

In [ ]:
dff = pd.DataFrame({'A': np.arange(10), 'B': list('aabbbbcccd')})
dff

In [ ]:
dff.groupby('B').filter(lambda x: len(x) > 2)

Alternatively, we can return a like-indexed objects where the groups that do not pass the filter are filled with `NaNs`.

In [ ]:
dff.groupby('B').filter(lambda x: len(x) > 2, dropna=False)

For `DataFrames` with multiple columns we can specify a column as the filter criterion.

In [ ]:
dff['C'] = np.arange(10)
dff

In [ ]:
dff.groupby('B').filter(lambda x: x.size > 2)

In [ ]:
dff.groupby('B').filter(lambda x: x.C.size > 2)

## 5 Apply: General split-apply-combine

The most general-purpose `GroupBy` method is `apply`. `apply` will infer based from the result whether it should act as a aggregator, transformer, or filter, depending on exactly what it is passed onto it. Thus, the grouped column(s) may be included in the output or not.

Let's go back to our NBA 2023 Regular Season dataset.

In [ ]:
data = pd.read_csv(
    'data/nba-2023-regular.csv', sep=';', encoding = 'ISO-8859-1',
).drop_duplicates(subset='Rk').set_index('Rk')
data

Say we want to get the top players (in terms of any stat), we can define a function as,

In [ ]:
def top_players(df, n=5, column='PTS'):
    return df.sort_values(column, ascending=False)[:n]

If we won't be doing any grouping, we can apply this function directly to our data frame.

In [ ]:
top_players(data)

We can group by teams and get:

In [ ]:
data.groupby('Tm').apply(top_players)

If you have a function that takes in other arguments or keywords, you can pass these after the function.

In [ ]:
data.groupby('Tm').apply(top_players, n=1, column='BLK')

Notice that the resulting object has a hierarchical index formed form the group keys. you can disable this behavior by passing `group_keys` as `False`.

In [ ]:
data.groupby('Tm', group_keys=False).apply(top_players, n=1)

Combining the `pd.qcut` function allows us to perform quantile and bucket analysis.

For example, if we divide the NBA players in terms of their age, we can get an aggregated statistics based on the resulting bucket categorization.

In [ ]:
data.groupby(pd.qcut(data.Age, 4)).PTS.agg(['mean', 'std'])

Another application of the `apply` is to use it in random sampling and permutation. For example, in MOnte Carlo simulation purposes.

Let's demonstrate this by creating a deck of playing cards.

In [ ]:
suits = ['H', 'S', 'C', 'D']
card_val = (list(range(1, 11)) + [10]*3) * 4
base_names = ['A'] + list(range(2, 11)) + ['J', 'K', 'Q']
cards = [str(num) + suit for suit in suits for num in base_names]

deck = pd.Series(card_val, index=cards)
deck.head(12)

We can define a function that draws a card from this deck randomly:

In [ ]:
def draw(deck, n=5):
    return deck.sample(n)

In [ ]:
draw(deck)

Suppose you wanted two random cards from each suit, since we define the suit as the last character of each name, we can define a function that retrieves it.

In [ ]:
def get_suit(card):
    return card[-1]

In [ ]:
deck.groupby(get_suit).apply(draw, n=2)

Alternatively, we could pass `group_keys` to be `False` to drop the outer suit index, leaving in just the selected cards.

In [ ]:
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

## 6 Additional Resources

Group By Codesolid - https://codesolid.com/pandas-groupby/

Practice Exercises - https://www.practiceprobs.com/problemsets/python-pandas/

Practice Exercises - https://www.w3resource.com/python-exercises/pandas/groupby/index.phpGroup